In [1]:
import psi4
import pandas as pd
import os
import numpy as np
from lps_uscf import lps_solver

In [2]:
csv_file = 'open_shell_15atoms_vs_uhf.csv'

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"-> Loaded existing results: {len(df)} rows found.")
else:
    df = pd.DataFrame()
    print("-> No existing file found. Starting fresh.")

-> Loaded existing results: 139 rows found.


In [69]:
TP = ['LDA_K_TF', 1.0]
LAMBDA = 0.166666
EXC = ['GGA_X_PBE', 0.0, 'LDA_C_VWN', 0.0]
FA = [True, 1.0]
DIIS = True
MAX_ITER = 10000
DAMPING = [0.99, 0.9, 0.00009]
# D_guess = [GUESS_A, GUESS_B]
D_guess = None
verbose=True

psi4.set_options({'basis': 'UGBS_S', 
                  'DFT_SPHERICAL_POINTS': 6, 
                  'DFT_RADIAL_POINTS': 1000})

mol = psi4.geometry("""
units bohr
0 2
K
symmetry c1
""")

E, Da, Db, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,mol,DAMPING,FA,D_guess,DIIS,verbose)
print('\nFinal SCF energy: %.4f Hartree' % E)

Number of basis functions:   31

Starting SCF iterations:

    Iter            Energy            Delta E         dRMS

SCF Iter  1:      9885.51092381    9.88551E+03    7.52108E+02
SCF Iter  2:      9673.03853755   -2.12472E+02    7.38308E+02
SCF Iter  3:      9464.64085499   -2.08398E+02    7.24755E+02
SCF Iter  4:     10582.08796164    1.11745E+03    8.49682E+02
SCF Iter  5:     11697.55195311    1.11546E+03    9.03339E+02
SCF Iter  6:     12943.89266588    1.24634E+03    9.49197E+02
SCF Iter  7:     14340.13937367    1.39625E+03    9.97784E+02
SCF Iter  8:     14078.29658164   -2.61843E+02    9.82051E+02
SCF Iter  9:     13820.33541420   -2.57961E+02    9.66862E+02
SCF Iter 10:     13565.32940828   -2.55006E+02    9.51817E+02
SCF Iter 11:     13314.79762513   -2.50532E+02    9.37348E+02
SCF Iter 12:     13061.79249431   -2.53005E+02    9.23137E+02
SCF Iter 13:     12813.37338043   -2.48419E+02    9.09525E+02
SCF Iter 14:     12572.43989643   -2.40933E+02    8.95461E+02
SCF Iter 15: 

In [70]:
GUESS_A = Da
GUESS_B = Db

In [3]:
psi4.core.set_output_file('output.dat', False)

ATOMS = {
    # Period 3 (Na-Ar)
    'Na': {'mult': 2},  # [Ne] 3s1
    'Mg': {'mult': 1},  # [Ne] 3s2
    'Al': {'mult': 2},  # [Ne] 3s2 3p1
    'Si': {'mult': 3},  # [Ne] 3s2 3p2 
    # 'P':  {'mult': 4},  # [Ne] 3s2 3p3 
    # 'S':  {'mult': 3},  # [Ne] 3s2 3p4
    # 'Cl': {'mult': 2},  # [Ne] 3s2 3p5
    # 'Ar': {'mult': 1},  # [Ne] 3s2 3p6
    # Period 4 (Selected)
    # 'K':  {'mult': 2},  # [Ar] 4s1
    # 'Ca': {'mult': 1},  # [Ar] 4s2
    # 'Cu': {'mult': 2},  # [Ar] 3d10 4s1 
    # 'Zn': {'mult': 1},  # [Ar] 3d10 4s2
    # 'Kr': {'mult': 1},  # [Ar] 3d10 4s2 4p6
}

METHOD = "TF0.111111W LDA"
TP = ['LDA_K_TF', 1.0]
LAMBDA = 0.111111
EXC = ['LDA_X', 1.0, 'LDA_C_VWN', 0.0]
# EXC = ['GGA_X_PBE', 1.0, 'LDA_C_VWN', 0.0]
FA = [False, 1.0]
DIIS = True
MAX_ITER = 15000
DAMPING = [0.99, 0.99, 0.00003]
# D_guess = [GUESS_A, GUESS_B]
D_guess = None
verbose=True

psi4.set_options({'basis': 'UGBS_S', 
                  'DFT_SPHERICAL_POINTS': 6, 
                  'DFT_RADIAL_POINTS': 1000})

for atom in ATOMS:
    
    if not df.empty:
        exists = df[
            (df['Atom'] == atom) & 
            (df['Method'] == METHOD) & 
            (df['Basis'] == psi4.core.get_global_option("BASIS"))
        ]
        if not exists.empty:
            print(f"Skipping {atom} (Already exists for {METHOD}/{psi4.core.get_global_option("BASIS")})")
            continue

    print(f"Calculating {atom} with {METHOD}...")
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    try:
        E, Da, Db, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,MOL,DAMPING,FA,D_guess,DIIS,verbose)
        if iterations >= MAX_ITER:
            print("  !!! SCF failed to converge (Max cycles exceeded).")
        else:
            print(f"Calculated Energy: {E:.4f} Hartree")
            row = {
                "Method": METHOD,
                "Atom": atom,
                "Basis": psi4.core.get_global_option("BASIS"),
                "Grid_Sph": psi4.core.get_global_option("DFT_SPHERICAL_POINTS"),
                "Grid_Rad": psi4.core.get_global_option("DFT_RADIAL_POINTS"),
                "Energy,Ha": round(E, 6),
                "Iterations": iterations,
                "DIIS": DIIS,
                "Damp_Start": DAMPING[0],
                "Damp_End": DAMPING[1],
                "Damp_Cutoff": DAMPING[2]
            }
            
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    
    except Exception as e:
        print(f"  !!! Failed {atom}. Error: {e}")
        continue

Calculating Na with TF0.111111W LDA...
Number of basis functions:   31

Starting SCF iterations:

    Iter            Energy            Delta E         dRMS

SCF Iter  1:      1013.66449694    1.01366E+03    9.26732E+01
SCF Iter  2:       986.95713182   -2.67074E+01    9.09041E+01
SCF Iter  3:       960.80417751   -2.61530E+01    8.91681E+01
SCF Iter  4:       935.99791645   -2.48063E+01    8.78138E+01
SCF Iter  5:       911.68973227   -2.43082E+01    8.71928E+01
SCF Iter  6:       887.88597139   -2.38038E+01    8.72969E+01
SCF Iter  7:       864.57851009   -2.33075E+01    8.80979E+01
SCF Iter  8:       841.44325652   -2.31353E+01    8.66941E+01
SCF Iter  9:       824.84699834   -1.65963E+01    8.51342E+01
SCF Iter 10:       803.45991924   -2.13871E+01    8.36134E+01
SCF Iter 11:       782.79111186   -2.06688E+01    8.21215E+01
SCF Iter 12:       761.05745521   -2.17337E+01    8.06503E+01
SCF Iter 13:       740.26518488   -2.07923E+01    7.92054E+01
SCF Iter 14:       719.59235259   -2

In [4]:
df.to_csv(csv_file, index=False)
df

,Method,Atom,Basis,Grid_Sph,Grid_Rad,"Energy,Ha",Iterations,DIIS,Damp_Start,Damp_End,Damp_Cutoff
0,TFW LDA,Na,UGBS_S,6,1000,-108.912083,284,True,0.90,0.90,0.00009
1,TFW LDA,Mg,UGBS_S,6,1000,-135.553609,237,True,0.90,0.90,0.00009
2,TFW LDA,Al,UGBS_S,6,1000,-165.669941,387,True,0.90,0.90,0.00009
3,TFW LDA,Si,UGBS_S,6,1000,-199.392286,517,True,0.90,0.90,0.00009
4,TFW LDA,P,UGBS_S,6,1000,-236.833553,342,True,0.90,0.90,0.00009
...,...,...,...,...,...,...,...,...,...,...,...
137,TF0.2W FA,Cl,UGBS_S,6,1000,-445.209435,4560,True,0.99,0.90,0.00003
138,TF0.166666W FA,Cl,UGBS_S,6,1000,-456.106275,10340,True,0.99,0.90,0.00003
139,TF0.111111W LDA,Na,UGBS_S,6,1000,-175.220390,4870,True,0.99,0.99,0.00003
140,TF0.111111W LDA,Mg,UGBS_S,6,1000,-215.265035,4402,True,0.99,0.99,0.00003


In [ ]:
## TF0.111111W PBE for O and F used TF0.111111W FA densities of O and F as initial guesses
## TF0.111111W FA for Si, Kr used TF0.166666W FA initial guess
## TF0.2W FA for Cl used TF0.333333W FA initial guess
## TF0.166666W FA for Cl used TF0.2W FA initial guess

In [22]:
ATOMS = {
    'He':  {'mult': 1}, 
    'Be': {'mult': 1},
    'Ne': {'mult': 1}, 
    'Mg': {'mult': 1},
    'Ar':  {'mult': 1}, 
    'Ca':  {'mult': 1},
    'Zn':  {'mult': 1}, 
    'Kr':  {'mult': 1}
}
psi4.core.set_output_file('output.dat', False)
psi4.set_options({'basis': 'UGBS',
                  'scf_type': 'PK'})
rhf_energies = {}
rhf_homos = {}
for atom in ATOMS:
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    E, wfn = psi4.energy('SCF', return_wfn=True)
    homo = wfn.epsilon_a().np[wfn.nalpha()-1]
    rhf_energies[atom] = round(E, 6)
    rhf_homos[atom] = round(homo, 6)
    print(f"RHF/UGBS {atom} Energy: {E:.4f} Hartree, IP: {homo:.4f}")

RHF/UGBS He Energy: -2.8617 Hartree, IP: -0.9180
RHF/UGBS Be Energy: -14.5730 Hartree, IP: -0.3093
RHF/UGBS Ne Energy: -128.5471 Hartree, IP: -0.8504
RHF/UGBS Mg Energy: -199.6146 Hartree, IP: -0.2530
RHF/UGBS Ar Energy: -526.8175 Hartree, IP: -0.5910
RHF/UGBS Ca Energy: -676.7582 Hartree, IP: -0.1955
RHF/UGBS Zn Energy: -1777.8481 Hartree, IP: -0.2925
RHF/UGBS Kr Energy: -2752.0549 Hartree, IP: -0.5242


In [26]:
atom_order = ['He', 'Be', 'Ne', 'Mg', 'Ar', 'Ca', 'Zn', 'Kr']
energy_table = df.pivot(index='Atom', columns='Method', values='Energy,Ha')
energy_table = energy_table.reindex(atom_order)
energy_table['RHF/UGBS'] = pd.Series(rhf_energies)
new_order = ['TFD0.166666W', 'TF0.166666W PBEx', 'TF0.166666W FA', 'RHF/UGBS']
energy_table = energy_table[new_order]

In [28]:
reference = energy_table['RHF/UGBS']

res = {}
for method, energies in energy_table.items():
    if method == 'RHF/UGBS': 
        continue 
    
    mae  = (energies - reference).abs().mean()
    rmae = (((energies - reference).abs()) / reference * -100).mean()
    res[method] = round(mae, 2), round(rmae, 2)

mae_row  = {method: values[0] for method, values in res.items()}
rmae_row = {method: values[1] for method, values in res.items()}
stats_df = pd.DataFrame([mae_row, rmae_row], index=['MAE(Ha)', 'rMAE(%)'])
energy_table = pd.concat([energy_table, stats_df], sort=False)

In [29]:
display(energy_table)

,TFD0.166666W,TF0.166666W PBEx,TF0.166666W FA,RHF/UGBS
He,-2.951247,-3.097069,-3.019526,-2.861680
Be,-15.040391,-15.388877,-14.660508,-14.573023
Ne,-132.508856,-133.573716,-128.291707,-128.547083
Mg,-204.540690,-205.864549,-198.305386,-199.614621
Ar,-537.208760,-539.346529,-522.928926,-526.817486
Ca,-690.396266,-692.814969,-672.808485,-676.758154
Zn,-1812.192490,-1816.067937,-1773.727670,-1777.848060
Kr,-2795.914635,-2800.696590,-2741.665221,-2752.054860
MAE(Ha),13.960000,15.970000,3.020000,NaN
rMAE(%),2.420000,3.690000,1.110000,NaN


In [33]:
atom_order = ['He', 'Be', 'Ne', 'Mg', 'Ar', 'Ca', 'Zn', 'Kr']
mu_table = df.pivot(index='Atom', columns='Method', values='ChemPot,Ha')
mu_table = mu_table.reindex(atom_order)
mu_table['RHF/UGBS'] = pd.Series(rhf_homos)
new_order = ['TFD0.166666W', 'TF0.166666W PBEx', 'TF0.166666W FA', 'RHF/UGBS']
mu_table = mu_table[new_order]

In [35]:
reference = mu_table['RHF/UGBS']

res = {}
for method, energies in mu_table.items():
    if method == 'RHF/UGBS': 
        continue 
    
    mae  = (energies - reference).abs().mean()
    rmae = (((energies - reference).abs()) / reference * -100).mean()
    res[method] = round(mae, 2), round(rmae, 2)

mae_row  = {method: values[0] for method, values in res.items()}
rmae_row = {method: values[1] for method, values in res.items()}
stats_df = pd.DataFrame([mae_row, rmae_row], index=['MAE(Ha)', 'rMAE(%)'])
mu_table = pd.concat([mu_table, stats_df], sort=False)

In [36]:
display(mu_table)

,TFD0.166666W,TF0.166666W PBEx,TF0.166666W FA,RHF/UGBS
He,-0.067106,-0.081193,-0.313483,-0.917956
Be,-0.069849,-0.081756,-0.272375,-0.309271
Ne,-0.072529,-0.082214,-0.231390,-0.850411
Mg,-0.072963,-0.082281,-0.224844,-0.253048
Ar,-0.073833,-0.082424,-0.211897,-0.590989
Ca,-0.074040,-0.082458,-0.208863,-0.195527
Zn,-0.074767,-0.082581,-0.198303,-0.292463
Kr,-0.075063,-0.082633,-0.194080,-0.524161
MAE(Ha),0.420000,0.410000,0.260000,NaN
rMAE(%),80.310000,77.800000,40.980000,NaN
